# Module 5 — Inverse Kinematics
## Unit 5 — Numerical Inverse Kinematics in Practice
### Lesson 5.4 — Numerical Inverse Kinematics in Practice (Unit 5 Recap)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Unit 5 synthesis
Pick the step rule by conditioning; both reach the target.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12, -L2*s12],[L1*c1+L2*c12, L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

def step_dls(J,e,lam): return J.T@np.linalg.inv(J@J.T+lam*lam*np.eye(len(e)))@e
def solve(target,theta0,L1,L2,lam_base=0.02,tol=1e-5,max_iter=200):
    theta=np.array(theta0,float); target=np.array(target,float)
    for it in range(max_iter):
        e=target-fk_two_link(theta[0],theta[1],L1,L2)
        if np.linalg.norm(e)<tol: return theta,it
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        detJ=abs(np.linalg.det(J))
        lam=lam_base if detJ>0.02 else 0.15   # more damping when ill-conditioned
        theta=theta+step_dls(J,e,lam)
    return theta,max_iter
L1,L2=0.4,0.3
sol,iters=solve([0.5,0.2],np.radians([10,20]),L1,L2)
print("adaptive-damping solve:",np.round(np.degrees(sol),2),"in",iters,"iters")


## Check your work


In [ ]:
sol,iters=solve([0.5,0.2],np.radians([10,20]),L1,L2)
assert np.allclose(fk_two_link(sol[0],sol[1],L1,L2),[0.5,0.2],atol=1e-4)
print("All checks passed.")
